<a href="https://colab.research.google.com/github/julianzamb/5-DoF-in-Colab/blob/main/5DoF_3D_SIMULATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q pythreejs
from google.colab import output
output.enable_custom_widget_manager()

In [5]:
import math, time, threading
from pythreejs import (
    Scene, PerspectiveCamera, Renderer, OrbitControls, Object3D, Mesh,
    BoxGeometry, CylinderGeometry, MeshStandardMaterial, AmbientLight,
    HemisphereLight, GridHelper, AxesHelper
)
from ipywidgets import FloatSlider, ToggleButton, VBox, HBox, Layout
from IPython.display import display

# ---------- Quaternion helpers (x, y, z, w) -- avoids Euler 'xyz' sync bug ----------
def q_x(a): return (math.sin(a/2), 0.0, 0.0, math.cos(a/2))
def q_y(a): return (0.0, math.sin(a/2), 0.0, math.cos(a/2))
def q_z(a): return (0.0, 0.0, math.sin(a/2), math.cos(a/2))

# ---------- Kinematic parameters (meters) ----------
BASE_H = 0.10
L0, L1, L2, L3, L4 = 0.20, 0.60, 0.50, 0.30, 0.20
W = 0.08

def mat(color):
    return MeshStandardMaterial(color=color, metalness=0.2, roughness=0.6)

def link(length, color, width=W):
    """Box spanning local z = 0 -> length."""
    return Mesh(BoxGeometry(width=width, height=width, depth=length), mat(color),
                position=(0, 0, length / 2))

def joint_cyl(axis, color, radius=0.06, length=0.12):
    m = Mesh(CylinderGeometry(radiusTop=radius, radiusBottom=radius,
                              height=length, radialSegments=24), mat(color))
    if axis == 'z':                      # cylinders are Y-aligned; turn Y -> Z
        m.quaternion = q_x(math.pi / 2)
    return m

# ---------- Kinematic chain ----------
base = Mesh(BoxGeometry(width=0.4, height=0.4, depth=BASE_H), mat('#444444'),
            position=(0, 0, BASE_H / 2))

j1 = Object3D(position=(0, 0, BASE_H))
j1.add(joint_cyl('z', '#cc3333'))
j1.add(link(L0, '#888888'))

j2 = Object3D(position=(0, 0, L0))
j2.add(joint_cyl('y', '#cc3333'))
j2.add(link(L1, '#3b82f6'))

j3 = Object3D(position=(0, 0, L1))
j3.add(joint_cyl('y', '#cc3333'))
j3.add(link(L2, '#22c55e'))

j4 = Object3D(position=(0, 0, L2))
j4.add(joint_cyl('y', '#cc3333'))
j4.add(link(L3, '#eab308'))

j5 = Object3D(position=(0, 0, L3))
j5.add(joint_cyl('z', '#cc3333', radius=0.05, length=0.06))
j5.add(link(L4, '#a855f7', width=0.05))

# ---------- Gripper (child of J5, so it rolls with Joint 5) ----------
# Frame sits at the J5 tip (z = L4). Fingers slide along the gripper's local X axis.
PLATE_T = 0.02                      # plate thickness (along Z)
FINGER_W, FINGER_D, FINGER_H = 0.02, 0.10, 0.03   # x-thickness, z-length, y-width
FINGER_Z = PLATE_T + FINGER_D / 2   # fingers stand on top of the plate
X_CLOSED, X_OPEN = 0.0105, 0.06     # closed: fingers nearly touch (1 mm gap, avoids z-fighting)

gripper = Object3D(position=(0, 0, L4))

plate = Mesh(BoxGeometry(width=0.16, height=0.05, depth=PLATE_T), mat('#222222'),
             position=(0, 0, PLATE_T / 2))
finger_l = Mesh(BoxGeometry(width=FINGER_W, height=FINGER_H, depth=FINGER_D),
                mat('#f97316'), position=(-X_CLOSED, 0, FINGER_Z))
finger_r = Mesh(BoxGeometry(width=FINGER_W, height=FINGER_H, depth=FINGER_D),
                mat('#f97316'), position=(X_CLOSED, 0, FINGER_Z))

gripper.add(plate); gripper.add(finger_l); gripper.add(finger_r)
j5.add(gripper)

base.add(j1); j1.add(j2); j2.add(j3); j3.add(j4); j4.add(j5)

# ---------- Scene (Z-up) ----------
grid = GridHelper(2, 20)
grid.quaternion = q_x(math.pi / 2)       # XZ plane -> XY plane (ground)
axes = AxesHelper(0.5)                   # X red, Y green, Z blue

camera = PerspectiveCamera(position=(1.8, 1.8, 1.4), up=(0, 0, 1),
                           fov=50, aspect=800/600)

scene = Scene(children=[
    base, grid, axes, camera,
    AmbientLight(color='#ffffff', intensity=0.5),
    HemisphereLight(skyColor='#ffffff', groundColor='#555555',
                    intensity=0.9, position=(0, 0, 1)),
], background='#f0f0f0')

controls = OrbitControls(controlling=camera, target=(0, 0, 0.7))
renderer = Renderer(camera=camera, scene=scene, controls=[controls],
                    antialias=True, width=800, height=600)

# ---------- Sliders ----------
def make_slider(name, lo=-180, hi=180):
    return FloatSlider(description=name, min=lo, max=hi, step=1, value=0,
                       continuous_update=True, readout_format='.0f',
                       layout=Layout(width='400px'))

s1 = make_slider('J1 yaw')
s2 = make_slider('J2 pitch', -90, 90)
s3 = make_slider('J3 pitch', -150, 150)
s4 = make_slider('J4 pitch', -120, 120)
s5 = make_slider('J5 roll')

def on_change(joint, axis):
    def cb(change):
        a = math.radians(change['new'])
        joint.quaternion = q_y(a) if axis == 'y' else q_z(a)
    return cb

s1.observe(on_change(j1, 'z'), names='value')
s2.observe(on_change(j2, 'y'), names='value')
s3.observe(on_change(j3, 'y'), names='value')
s4.observe(on_change(j4, 'y'), names='value')
s5.observe(on_change(j5, 'z'), names='value')

# ---------- Gripper toggle + animation (runs on the main kernel thread) ----------
grip = {'t': 0.0}                   # 0 = closed, 1 = open

def set_gripper(t):
    x = X_CLOSED + (X_OPEN - X_CLOSED) * t
    finger_l.position = (-x, 0, FINGER_Z)
    finger_r.position = (x, 0, FINGER_Z)

def animate_gripper(target, duration=0.35, steps=14):
    start = grip['t']
    for i in range(1, steps + 1):
        s = i / steps
        s = s * s * (3 - 2 * s)      # smoothstep easing
        grip['t'] = start + (target - start) * s
        set_gripper(grip['t'])
        time.sleep(duration / steps)
    # Guarantee the exact end pose, and resend it once in case a message was dropped
    grip['t'] = target
    set_gripper(target)
    time.sleep(0.05)
    set_gripper(target)

gripper_toggle = ToggleButton(value=False, description='Open/Close Gripper',
                              button_style='info', icon='hand-paper',
                              tooltip='Off = closed, On = open',
                              layout=Layout(width='400px'))

def on_toggle(change):
    animate_gripper(1.0 if change['new'] else 0.0)

gripper_toggle.observe(on_toggle, names='value')

set_gripper(0.0)                     # start closed

display(HBox([renderer, VBox([s1, s2, s3, s4, s5, gripper_toggle])]))